# **Analysis**

In this section, we go beyond the raw observations made in the previous phase and interpret the data in depth.<br>
While the observation phase focused on describing the dataset as it is, the analysis phase aims to understand what the data reveals and what implications it has for model training.

Each subsection presents a specific aspect of the dataset, supported by visualizations and followed by a detailed interpretation.<br>
The goal is to identify potential challenges, confirm assumptions, and justify the preprocessing decisions made in the clean phase.

This analysis covers the following aspects:

- **Class distribution**: Is the dataset balanced across all four cancer types ?
- **Train/Test split per class**: Is the 80/20 split consistent across classes ?
- **Corrupted images per class**: Are corrupted files isolated or class-specific ?
- **Color channels**: Do all classes share the same color mode ?

Each of these factors can have a direct impact on the performance and fairness of the model, and understanding them is essential before proceeding to training.

---

## **Class Distribution**

To ensure fair model training, it is important to verify that all four cancer types are equally represented in the dataset.<br>
An imbalanced dataset could cause the model to become biased toward the dominant class, leading to poor performance on underrepresented cancer types.

We analyze the class distribution using both a bar chart showing the absolute number of samples per class, and a percentage chart showing the relative proportion of each class in the dataset.

In [ ]:
from pathlib import Path
import sys

# Auto-detect project root — works regardless of where Jupyter was launched
_search = Path.cwd()
while not (_search / "supervised").exists() and _search != _search.parent:
    _search = _search.parent

PROJECT_ROOT    = _search
SUPERVISED_ROOT = PROJECT_ROOT / "supervised"
DATASET_DIR     = str(SUPERVISED_ROOT / "dataset")
CSV_PATH        = str(SUPERVISED_ROOT / "dataset" / "dataset_info.csv")
CLEAN_DIR       = str(SUPERVISED_ROOT / "clean_dataset")
CLEAN_CSV_PATH  = str(SUPERVISED_ROOT / "clean_dataset" / "dataset_info.csv")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"CSV          : {CSV_PATH}")

In [ ]:
from supervised.src.utils import load_class_counts
from supervised.src.graph import plot_class_distribution, plot_class_distribution_percent

if __name__ == "__main__":
    names, data = load_class_counts(CSV_PATH)
    plot_class_distribution(names, data)
    plot_class_distribution_percent(names, data)

### **Results Interpretation**

The dataset contains a total of **7,023 images** distributed across four classes:

| Class | Samples | Percentage |
|---|---|---|
| Glioma | 1,621 | 23.1% |
| Meningioma | 1,645 | 23.4% |
| No Tumor | 2,000 | 28.5% |
| Pituitary | 1,757 | 25.0% |

The distribution shows that all four classes are relatively well represented, with percentages ranging from **23.1% to 28.5%**.<br>
The ratio between the largest class (No Tumor, 2,000 samples) and the smallest class (Glioma, 1,621 samples) is approximately **1.23:1**, which is well below our imbalance threshold of 3:1.

This balanced distribution is a favorable condition for training, as it reduces the risk of the model becoming biased toward any specific cancer type.<br>
No resampling technique such as oversampling or class weighting is required, and the model can be expected to learn equally well across all four categories.

---

## **Train/Test Split per Class**

The global dataset split of **81.3% training** and **18.7% testing** gives a general overview of how the data is divided.<br>
However, it is important to verify that this split is consistently applied across all four cancer types individually.

An uneven split per class could mean that certain cancer types are underrepresented in the test set, leading to unreliable evaluation results for those specific classes.<br>
This is particularly critical in a medical context, where poor performance on a specific cancer type could have serious consequences.

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

def plot_split_per_class(csv_path):
    classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

    train_counts = {}
    test_counts = {}

    with open(csv_path, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row['class_name']
            split = row['split']

            if label in classes:
                if split == 'Training':
                    train_counts[label] = train_counts.get(label, 0) + 1
                elif split == 'Testing':
                    test_counts[label] = test_counts.get(label, 0) + 1

    train_values = [train_counts[cls] for cls in classes]
    test_values = [test_counts[cls] for cls in classes]

    x = np.arange(len(classes))
    width = 0.35

    # Plot
    plt.figure(figsize=(8, 5))
    plt.bar(x - width / 2, train_values, width, label='Training')
    plt.bar(x + width / 2, test_values, width, label='Testing')

    # Labels
    plt.xlabel('Classes')
    plt.ylabel('Number of samples')
    plt.title("Repartition Training and Testing per class")
    plt.xticks(x, classes)
    plt.legend()

    # Print text
    for i in range(len(classes)):
        plt.text(x[i] - width / 2, train_values[i], str(train_values[i]), ha='center', va='bottom')
        plt.text(x[i] + width / 2, test_values[i], str(test_values[i]), ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_split_per_class(CLEAN_CSV_PATH)

### **Results Interpretation**

The train/test split per class reveals the following distribution:

| Class | Training | Testing | Train % |
|---|---|---|---|
| Glioma | 1,321 | 300 | 81.5% |
| Meningioma | 1,339 | 306 | 81.4% |
| No Tumor | 1,595 | 405 | 79.8% |
| Pituitary | 1,457 | 300 | 82.9% |

The split is consistently applied across all four classes, with training proportions ranging from **79.8% to 82.9%**, all close to the global **81.3% / 18.7%** split.

No class is significantly over or underrepresented in either the training or testing set. This ensures that the model will be trained and evaluated fairly across all four cancer types, without any class being disadvantaged during evaluation.

---

## **Corrupted Images per Class**

During the observation phase, we verified the overall integrity of the dataset and confirmed that no corrupted images were detected globally.<br>
However, it is important to go further and verify whether corrupted images are isolated or concentrated in a specific class.

In a medical context, corrupted images are particularly problematic.<br>
A corrupted image cannot be read or processed correctly, which can cause errors during training and lead to incomplete learning for the affected class.<br>
If corrupted images were concentrated in a single cancer type, the model would have significantly less data to learn from for that specific class, potentially leading to poor detection performance for that cancer type.<br>
In a real-world medical scenario, this could result in missed diagnoses, which can have serious consequences for patients.<br>

For these reasons, ensuring that no class contains corrupted images is an essential step before proceeding to model training.

In [ ]:
from supervised.src.graph import plot_corrupted_images

if __name__ == "__main__":
    plot_corrupted_images(CLEAN_CSV_PATH)

### **Results Interpretation**

The analysis confirms that no corrupted images were detected in any
of the four classes:

| Class | Corrupted Images |
|---|---|
| Glioma | 0 |
| Meningioma | 0 |
| No Tumor | 0 |
| Pituitary | 0 |

Every class retains its full set of images, meaning no data loss occurred in any category.<br>
The dataset is fully intact and each cancer type is equally preserved, ensuring that the model will have access to the complete data for every class during training.

---

## **Color Channels per Class**

During the observation phase, we identified that some images in the dataset were not in RGB format, with some using RGBA or other color modes.<br>
While the clean phase converted all images to RGB format, it is important to verify that this conversion was applied consistently across all four classes.

An inconsistency in color modes between classes could introduce bias in the model, as images with different numbers of channels would not be processed in the same way. In a medical context, this is particularly important as the model must treat all cancer types equally.<br>

We analyze the color mode distribution per class to confirm that all images now share the same **RGB** format with exactly **3 color channels**.<br>

In [ ]:
import csv
import matplotlib.pyplot as plt

from PIL import Image


def plot_colors_channels_per_classes(csv_path: str, directory: str):
    class_modes = {}
    all_modes = set()

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            split = row["split"]
            class_name = row["class_name"]
            filename = row["filename"]
            path = f"{directory}/{split}/{class_name}/{filename}"
            if class_name not in class_modes:
                class_modes[class_name] = {}
            try:
                with Image.open(path) as img:
                    mode = img.mode
                    all_modes.add(mode)
                    class_modes[class_name][mode] = class_modes[class_name].get(mode, 0) + 1
            except:
                pass
    classes = list(class_modes.keys())
    modes = list(all_modes)
    x = range(len(classes))
    width = 0.8 / len(modes)

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, mode in enumerate(modes):
        values = [class_modes[cls].get(mode, 0) for cls in classes]
        offset = [j + i * width - (len(modes) - 1) * width / 2 for j in x]
        bars = ax.bar(offset, values, width, label=mode)

        for bar, value in zip(bars, values):
            if value > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(value), ha="center", va="bottom")

    ax.set_xticks(x)
    ax.set_xticklabels(classes)
    ax.set_ylabel("Number of Images")
    ax.set_title("Color Channels per Class")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    plot_colors_channels_per_classes(
        csv_path=CLEAN_CSV_PATH,
        directory=CLEAN_DIR
    )

### **Results Interpretation**

The analysis confirms that all images across every class are now in **RGB** format:

| Class | RGB | Grayscale | Other |
|---|---|---|---|
| Glioma | 1,621 | 0 | 0 |
| Meningioma | 1,645 | 0 | 0 |
| No Tumor | 2,000 | 0 | 0 |
| Pituitary | 1,757 | 0 | 0 |

Every single image in the dataset has been successfully converted to RGB format during the clean phase. No class contains any grayscale or other color mode images, confirming that the conversion was applied consistently and correctly across the entire dataset.

This uniformity ensures that all images will be processed identically during training, with exactly **3 color channels** per image, eliminating any risk of inconsistency caused by mixed color modes.

---

# Analysis — Conclusion

Through this analysis phase, we have examined the dataset from multiple angles to identify potential challenges and confirm the quality of the data before proceeding to model training.

The key findings are the following:

- **Class distribution**: The dataset is well balanced across all four cancer types, with proportions ranging from **23.1% to 28.5%** and a ratio of **1.23:1** between the largest and smallest class. No resampling technique is required.

- **Train/Test split**: The 80/20 split is consistently applied across all
  four classes, with training proportions ranging from **79.8% to 82.9%**.
  The model will be trained and evaluated fairly across all cancer types.

- **Corrupted images**: No corrupted images were detected in any class,
  confirming that the dataset is fully intact and no data loss occurred.

- **Color channels**: All images are now uniformly in **RGB** format across
  all classes, confirming that the clean phase was applied correctly and
  consistently.

Overall, the dataset is well structured, balanced, and consistent.<br>
The preprocessing steps applied during the clean phase have successfully addressed all issues identified during the observation phase.<br>
The dataset is now fully ready for model training.